2) Parallelization

What is Parallelization in LangGraph?

In LangGraph, nodes typically execute in a sequence defined by edges, but when tasks don’t depend on each other’s outputs, you can run them in parallel. This is achieved by:

• Defining multiple nodes that can operate independently.
• Connecting them to a common starting point (e.g., START or another node).
• Merging their outputs into a downstream node if needed.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen-2.5-32b")
result=llm.invoke("Write a short story about a robot learning to love.")

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# Graph state
class State(TypedDict):
    topic: str
    characters: str
    settings: str
    premises: str
    story_intro: str

In [ ]:
# Nodes
def generate_characters(state: State):
    """Generate character descriptions"""
    msg = llm.invoke(
        f"Create two character names and brief traits for a story about {state['topic']}"
    )
    return {"characters": msg.content}

def generate_setting(state: State):
    """Generate a story setting"""
    msg = llm.invoke(
        f"Describe a vivid setting for a story about {state['topic']}"
    )
    return {"settings": msg.content}


def generate_premise(state: State):
    """Generate a story premise"""
    msg = llm.invoke(
        f"Write a one-sentence plot premise for a story about {state['topic']}"
    )
    return {"premises": msg.content}

def combine_elements(state: State):
    """Combine characters, setting, and premise into an intro"""
    msg = llm.invoke(
        f"Write a short story introduction using these elements:\n"
        f"Characters: {state['characters']}\n"
        f"Setting: {state['settings']}\n"
        f"Premise: {state['premises']}"
    )
    return {"story_intro": msg.content}

In [ ]:
# Build the graph
graph = StateGraph(State)

graph.add_node("character", generate_characters)
graph.add_node("setting", generate_setting)
graph.add_node("premise", generate_premise)
graph.add_node("combine", combine_elements)

In [ ]:
graph.add_edge(START, "character")
graph.add_edge(START, "setting")
graph.add_edge(START, "premise")

graph.add_edge("character", "combine")
graph.add_edge("setting", "combine")
graph.add_edge("premise", "combine")

graph.add_edge("combine", END)

# Compile and run
compiled_graph = graph.compile()

graph_image = compiled_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))

In [ ]:
state = {"topic": "time travel"}
result = compiled_graph.invoke(state)
print(result["story_intro"])